In [1]:
import sys
import os

# Get the absolute path to the scripts directory
module_path = os.path.abspath(os.path.join('scripts'))  # Relative to the notebook's location

# Add it to the system path if it's not already there
if module_path not in sys.path:
    sys.path.append(module_path)

In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
import triplet_functions
import init_gpu
import init_dataset
from train_vae import ConvVAE_BatchNorm, Sampling
from hyperplane import get_hyperplane

import numpy as np
import tensorflow as tf


def cross_location_synthesis(source_df, vae, domain_discriminator, target_location='LOC3', mean=0.1, std=0.1, n_samples=1, ):
    """
    Generates synthetic latent space samples by projecting embeddings onto a hyperplane 
    and perturbing them along the normal direction.

    Args:
        source_z (np.ndarray): Latent space embeddings of shape (batch_size, latent_dim)
        source_df (pd.DataFrame): DataFrame containing the original traces with 'Website' and 'Location' columns.
        w (np.ndarray): Normal vector of the hyperplane (latent_dim,)
        b (float): Bias term for the hyperplane.
        mean (float): Mean for Gaussian noise.
        std (float): Standard deviation for Gaussian noise.
        n_samples (int): Number of samples to generate per embedding.

    Returns:
        np.ndarray: Generated latent embeddings of shape (batch_size * n_samples, latent_dim)
        np.ndarray: Corresponding Website values for each generated sample.
        np.ndarray: Corresponding Location values for each generated sample.
    """

    w, b = get_hyperplane(domain_discriminator)

    # Get the latent space embeddings from the source DataFrame
    source_z = batched_encode(
        vae, source_df.iloc[:, 2:].to_numpy().astype(np.float32))

    batch_size, latent_dim = source_z.shape

    # normalize the normal vector
    w = w / np.linalg.norm(w)

    # Generate Gaussian noise (alpha_samples) for all embeddings at once
    # shape: (batch_size, n_samples, latent_dim)
    alpha_samples = np.random.normal(
        mean, std, (batch_size, n_samples, latent_dim))

    # Project all embeddings onto the hyperplane
    # shape: (batch_size, latent_dim)
    z_perpendicular = source_z - \
        (np.sum(source_z * w, axis=-1, keepdims=True) + b) * w

    # Generate n_samples for each embedding by adding perturbation along the normal vector
    # shape: (batch_size, n_samples, latent_dim)
    z_samples = z_perpendicular[:, None, :] + alpha_samples * w

    # Reshape the samples
    # shape: (batch_size * n_samples, latent_dim)
    z_samples_reshaped = z_samples.numpy().reshape(-1, latent_dim)

    # Create corresponding Website and Location (target) labels
    # shape: (batch_size * n_samples,)
    websites = np.repeat(source_df['Website'].values, n_samples)
    locations = np.repeat(target_location, batch_size * n_samples)

    # decode z_samples to get synthetic features
    synth_features = batched_decode(
        vae, z_samples_reshaped)

    synth_df = pd.DataFrame(synth_features, columns=source_df.columns[2:])
    # Insert Website and Location as the first two columns.
    synth_df.insert(0, 'Location', locations)
    synth_df.insert(0, 'Website', websites)

    return synth_df


def batched_encode(vae, x, batch_size=256):
    """
    Run vae.encode on x in smaller chunks to fit memory.
    Returns concatenated z_sample of shape (len(x), latent_dim).
    """
    z_list = []
    for i in range(0, len(x), batch_size):
        chunk = x[i:i+batch_size]
        _, _, z_chunk = vae.encode(chunk)
        z_list.append(z_chunk)
    return np.concatenate(z_list, axis=0)


def batched_decode(vae, z, batch_size=256):
    """
    Run vae.decode on z in smaller chunks.
    Returns concatenated reconstructions of shape (len(z), D).
    """
    x_list = []
    for i in range(0, len(z), batch_size):
        chunk = z[i:i+batch_size]
        x_chunk = vae.decode(chunk)
        x_list.append(x_chunk)
    return np.concatenate(x_list, axis=0)

In [10]:
init_gpu.initialize_gpus()
locations = ['LOC2', 'LOC3']
df = pd.read_csv(
    f"../dataset/processed/{locations[0]}-{locations[1]}-scaled-balanced.csv")
train_df, test_df, train_web_samples, test_web_samples = init_dataset.get_sample(
    df, locations, range(1500), 1200)
input_dim = train_df.shape[1] - 2

source_test_df = test_df[test_df['Location'] == locations[0]]
target_test_df = test_df[test_df['Location'] == locations[1]]


# load VAE
vae = tf.keras.models.load_model(f"../models-{locations[0]}-{locations[1]}/vae/ci_vae/ConvBased/domain_and_class/vae-e1000-mse1-kl0.0001-cl1.0-ldim96-hdim128.keras", custom_objects={
                                    'ConvVAE_BatchNorm': ConvVAE_BatchNorm, 'Sampling': Sampling})
vae.trainable = False  # freeze VAE weights
print("VAE loaded successfully!")
print(vae.summary())

domain_discriminator = tf.keras.models.load_model(
    f'../models-{locations[0]}-{locations[1]}/vae/ci_vae/ConvBased/domain_and_class/domain-discriminator-e1000.keras')
domain_discriminator.trainable = False  # freeze discriminator weights
print("Domain Discriminator loaded successfully!")
print(domain_discriminator.summary())


mean = 1.0
std = 0.1
print("Generating synthetic data...")
synth_df = cross_location_synthesis(
    source_test_df, vae, domain_discriminator, target_location=locations[1], n_samples=1, mean=mean, std=std)
print("Synthetic data generated successfully!")

Num GPUs Available:  0
Training Websites: [1309, 228, 51, 563, 501, 457, 285, 209, 1385, 1116, 178, 1209, 864, 65, 61, 191, 447, 476, 1034, 1232, 54, 1149, 407, 1466, 1330, 1436, 1490, 859, 451, 919, 1206, 569, 13, 326, 1429, 865, 696, 1468, 318, 440, 689, 1492, 189, 778, 198, 735, 704, 1236, 541, 88, 940, 1098, 255, 775, 161, 1130, 600, 1287, 1266, 740, 1182, 393, 142, 93, 1354, 466, 592, 163, 1482, 206, 1456, 1462, 928, 1301, 747, 333, 758, 727, 429, 1372, 546, 1399, 1327, 146, 1247, 1300, 350, 1093, 1495, 334, 946, 777, 552, 1310, 1140, 449, 1402, 664, 114, 469, 1486, 646, 821, 548, 135, 432, 1161, 644, 435, 1342, 1022, 810, 1316, 939, 292, 542, 1493, 505, 1478, 1103, 538, 1197, 877, 1195, 817, 741, 1404, 283, 1043, 1010, 186, 96, 224, 313, 1285, 327, 1487, 1221, 130, 788, 781, 1220, 958, 1083, 514, 1133, 23, 234, 1099, 1419, 1312, 1463, 1498, 601, 890, 323, 929, 6, 539, 1025, 365, 1039, 217, 1280, 611, 1308, 1338, 1415, 1477, 1366, 765, 330, 1104, 1086, 1, 1226, 663, 1000, 39, 229,

c:\Users\kaush\Documents\Python Projects\DoH-Synthesis\code\scripts\init_dataset.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df.sort_values(by=["Location"], inplace=True)
c:\Users\kaush\pyenv\ml_env\Lib\site-packages\keras\src\layers\core\input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


VAE loaded successfully!


Model: "conv_vae__batch_norm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential_8 (Sequential)       │ (None, 128)            │       198,144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sampling_4 (Sampling)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_9 (Sequential)       │ (None, 128)            │       227,072 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_10 (Sequential)      │ (None, 8192)           │       659,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (128, 96)              │       786,528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (128, 96)              │       786,528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sampling_5 (Sampling)           │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_11 (Sequential)      │ (None, 128)            │     1,684,741 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_10 (Dense)                │ (None, 2)              │           194 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 194 (776.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 194 (776.00 B)

None
Generating synthetic data...
Synthetic data generated successfully!


In [11]:
# Generate Kernel Density Estimation (KDE) Plots for the source_test_df and synth_df
from concurrent.futures import ThreadPoolExecutor
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

def _plot_for_website(website, source_df, target_df, synth_df, feature_cols,
                      output_dir, num_sample_overlays):
    """Worker: build & save one 1×3 KDE comparison for a single website."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
    fig.suptitle(f'KDE Comparison — Website: {website}', fontsize=16)

    for ax, (df, label) in zip(axes, [
        (source_df, "Source"),
        (target_df, "Target"),
        (synth_df,  "Synthetic")
    ]):
        subset = df[df['Website'] == website]
        if subset.shape[0] < 2:
            ax.text(0.5, 0.5, "Not enough samples", ha="center", va="center")
            ax.set_title(label)
            ax.set_xlabel("Packet Count")
            continue

        agg = subset[feature_cols].values.flatten()
        sns.kdeplot(agg, ax=ax, fill=True, alpha=0.6, linewidth=2)

        # optional individual-sample overlays
        if num_sample_overlays > 0:
            samp = subset.sample(min(num_sample_overlays, len(subset)),
                                 random_state=42)
            for _, row in samp.iterrows():
                sns.kdeplot(row[feature_cols].values,
                            ax=ax,
                            linestyle='--',
                            alpha=0.3,
                            linewidth=1)

        ax.set_title(label)
        ax.set_xlabel("Packet Count")

    axes[0].set_ylabel("Density")
    plt.tight_layout(rect=[0, 0, 1, 0.95])

    outpath = os.path.join(output_dir, f"kde_cmp_{website}.png")
    plt.savefig(outpath)
    plt.close(fig)
    return f"Saved comparison for {website} → {outpath}"

def plot_kde_comparison_per_class(source_df, target_df, synth_df, websites, 
                                     output_dir="../figures/kde_plots",
                                     num_sample_overlays=0,
                                     max_workers=4):
    """
    Parallelized KDE comparison:
      - One concurrent task per class (Website).
      - Saves each as a 1×3 subplot figure in output_dir.
    """
    os.makedirs(output_dir, exist_ok=True)

    feature_cols = source_df.columns[2:]
    # websites = source_df['Website'].unique()

    # Launch a thread per website
    with ThreadPoolExecutor(max_workers=max_workers) as exe:
        futures = [
            exe.submit(_plot_for_website,
                       website,
                       source_df,
                       target_df,
                       synth_df,
                       feature_cols,
                       output_dir,
                       num_sample_overlays)
            for website in websites
        ]
        for fut in futures:
            print(fut.result())


In [ ]:

# print("Plotting KDEs for Source Test Data...")
# plot_kde_per_class_domain(source_test_df, "Source Test")

# print("Plotting KDEs for Target Test Data...")
# plot_kde_per_class_domain(target_test_df, "Target Test")

# print("Plotting KDEs for Synthetic Data...")
# plot_kde_per_class_domain(synth_df, "Synthetic")


Plotting KDEs for Source Test Data...
Plotting KDEs for Target Test Data...
Plotting KDEs for Synthetic Data...


In [14]:
plot_kde_comparison_per_class(
    source_test_df,
    target_test_df,
    synth_df,
    test_web_samples[:10],
    max_workers=1,
    output_dir=f"../figures/kde_plots/mean-{mean}-std-{std}/",
    num_sample_overlays=0  # or 0 if you don’t want the individual-sample overlays
)

Saved comparison for 513 → ../figures/kde_plots/mean-1.0-std-0.1/kde_cmp_513.png
Saved comparison for 515 → ../figures/kde_plots/mean-1.0-std-0.1/kde_cmp_515.png
Saved comparison for 1028 → ../figures/kde_plots/mean-1.0-std-0.1/kde_cmp_1028.png
Saved comparison for 1030 → ../figures/kde_plots/mean-1.0-std-0.1/kde_cmp_1030.png
Saved comparison for 8 → ../figures/kde_plots/mean-1.0-std-0.1/kde_cmp_8.png
Saved comparison for 9 → ../figures/kde_plots/mean-1.0-std-0.1/kde_cmp_9.png
Saved comparison for 520 → ../figures/kde_plots/mean-1.0-std-0.1/kde_cmp_520.png
Saved comparison for 11 → ../figures/kde_plots/mean-1.0-std-0.1/kde_cmp_11.png
Saved comparison for 1032 → ../figures/kde_plots/mean-1.0-std-0.1/kde_cmp_1032.png
Saved comparison for 1033 → ../figures/kde_plots/mean-1.0-std-0.1/kde_cmp_1033.png


# Dimensionality Reduction

In [20]:
import os
import numpy as np
import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor
from sklearn.decomposition import PCA

def _plot_pca_for_website(website, source_df, target_df, synth_df,
                          feature_cols, output_dir):
    """Worker: build & save one 1×4 PCA comparison for a single website."""
    # subsets
    src = source_df[source_df['Website'] == website][feature_cols].values
    tgt = target_df[target_df['Website'] == website][feature_cols].values
    syn = synth_df[synth_df['Website'] == website][feature_cols].values

    # need at least 2 samples per domain
    if any(arr.shape[0] < 2 for arr in (src, tgt, syn)):
        return f"Skipping {website}: not enough samples"

    # Prepare figure
    fig, axes = plt.subplots(1, 4, figsize=(24, 5), sharex=False, sharey=False)
    fig.suptitle(f'PCA Comparison — Website: {website}', fontsize=18)

    # 1) Source-only PCA
    pca_s = PCA(n_components=2).fit(src)
    src_p = pca_s.transform(src)
    axes[0].scatter(src_p[:,0], src_p[:,1], alpha=0.7, s=20)
    axes[0].set_title("Source PCA")
    axes[0].set_xlabel("PC1"); axes[0].set_ylabel("PC2")

    # 2) Target-only PCA
    pca_t = PCA(n_components=2).fit(tgt)
    tgt_p = pca_t.transform(tgt)
    axes[1].scatter(tgt_p[:,0], tgt_p[:,1], alpha=0.7, s=20)
    axes[1].set_title("Target PCA")
    axes[1].set_xlabel("PC1"); axes[1].set_ylabel("")

    # 3) Synthetic-only PCA
    pca_y = PCA(n_components=2).fit(syn)
    syn_p = pca_y.transform(syn)
    axes[2].scatter(syn_p[:,0], syn_p[:,1], alpha=0.7, s=20)
    axes[2].set_title("Synthetic PCA")
    axes[2].set_xlabel("PC1"); axes[2].set_ylabel("")

    # 4) Common PCA on all data, colored by domain
    all_data = np.vstack([src, tgt, syn])
    domains = (["Source"] * src.shape[0] +
               ["Target"] * tgt.shape[0] +
               ["Synthetic"] * syn.shape[0])
    pca_all = PCA(n_components=2).fit(all_data)
    all_p = pca_all.transform(all_data)

    for dom, color in zip(["Source","Target","Synthetic"], ["C0","C1","C2"]):
        mask = np.array(domains) == dom
        axes[3].scatter(all_p[mask,0], all_p[mask,1],
                        alpha=0.7, s=20, label=dom, c=color)
    axes[3].set_title("Common PCA")
    axes[3].set_xlabel("PC1"); axes[3].set_ylabel("")
    axes[3].legend()

    plt.tight_layout(rect=[0, 0, 1, 0.92])

    # save
    os.makedirs(output_dir, exist_ok=True)
    outpath = os.path.join(output_dir, f"pca_cmp_{website}.png")
    plt.savefig(outpath)
    plt.close(fig)
    return f"Saved PCA comparison for {website} → {outpath}"

def plot_pca_comparison_per_class_mt(source_df, target_df, synth_df, websites,
                                     output_dir="../figures/pca_plots",
                                     max_workers=4):
    """Parallelized 1×4 PCA comparison per class."""
    feature_cols = source_df.columns[2:]
    # websites = source_df['Website'].unique()

    with ThreadPoolExecutor(max_workers=max_workers) as exe:
        futures = [
            exe.submit(_plot_pca_for_website,
                       website,
                       source_df,
                       target_df,
                       synth_df,
                       feature_cols,
                       output_dir)
            for website in websites
        ]
        for fut in futures:
            print(fut.result())

In [21]:
plot_pca_comparison_per_class_mt(
    source_test_df,
    target_test_df,
    synth_df,
    test_web_samples[:10],
    output_dir=f"../figures/pca_plots/mean-{mean}-std-{std}/",
    max_workers=1
)

Saved PCA comparison for 513 → ../figures/pca_plots/mean-1.0-std-0.1/pca_cmp_513.png
Saved PCA comparison for 515 → ../figures/pca_plots/mean-1.0-std-0.1/pca_cmp_515.png
Saved PCA comparison for 1028 → ../figures/pca_plots/mean-1.0-std-0.1/pca_cmp_1028.png
Saved PCA comparison for 1030 → ../figures/pca_plots/mean-1.0-std-0.1/pca_cmp_1030.png
Saved PCA comparison for 8 → ../figures/pca_plots/mean-1.0-std-0.1/pca_cmp_8.png
Saved PCA comparison for 9 → ../figures/pca_plots/mean-1.0-std-0.1/pca_cmp_9.png
Saved PCA comparison for 520 → ../figures/pca_plots/mean-1.0-std-0.1/pca_cmp_520.png
Saved PCA comparison for 11 → ../figures/pca_plots/mean-1.0-std-0.1/pca_cmp_11.png
Saved PCA comparison for 1032 → ../figures/pca_plots/mean-1.0-std-0.1/pca_cmp_1032.png
Saved PCA comparison for 1033 → ../figures/pca_plots/mean-1.0-std-0.1/pca_cmp_1033.png
